# Bài 3: Pipeline làm sạch & hợp nhất đa nguồn, kèm báo cáo chất lượng dữ liệu

**Học phần:** Khai phá dữ liệu (Data Mining)  
**Nguồn dữ liệu duy nhất:** `Chapter_4/data` (gồm `manifest.csv` và các tệp replay JSON `*.json` mô phỏng môi trường game **Kaggriculture**).

Notebook này trình bày giải pháp hoàn chỉnh, độc lập và có khả năng tái sử dụng cao cho 6 yêu cầu kỹ thuật:
1. **Hợp nhất đa nguồn & Khử trùng lặp qua thời gian (De-duplicate over time)**
2. **Chuẩn hóa đa phiên bản engine & Đa cấu hình game**
3. **Phát hiện bất thường cấp Episode và Turn (Rule-based vs Thống kê)**
4. **Xử lý dữ liệu thiếu có cấu trúc (Structured Missingness)**
5. **Báo cáo chất lượng dữ liệu, Ma trận truy vết (Traceability) & Đánh giá tác động Downstream**
6. **Kiến trúc Pipeline module hóa, tái sử dụng được**


In [2]:
from pathlib import Path
import json
import pandas as pd
import numpy as np

# Cấu hình đường dẫn hoàn toàn độc lập, chỉ dựa vào Chapter_4/data
ROOT = Path.cwd()
DATA_DIR = ROOT.parent / 'data'
OUTPUT_DIR = ROOT / 'output'

print('Thư mục dữ liệu nguồn:', DATA_DIR)
print('Thư mục xuất kết quả:', OUTPUT_DIR)


Thư mục dữ liệu nguồn: /home/nk/Projects/data-mining/Chapter_4/data
Thư mục xuất kết quả: /home/nk/Projects/data-mining/Chapter_4/Bai_3/output


## 1. Hợp nhất đa nguồn & Khử trùng lặp qua thời gian

### 1.1. Chứng minh Khóa hợp nhất chuẩn (`episode_id` vs UUID `id`)
Mỗi tệp JSON replay chứa:
* `info.EpisodeId`: Số nguyên định danh duy nhất của trận đấu (khớp với tên tệp `<episode_id>.json` và cột `episode_id` trong `manifest.csv`).
* `id`: Chuỗi UUID nội bộ của lần export replay (không tồn tại trong manifest).
* **Kết luận chứng minh:** Khóa hợp nhất đúng đắn duy nhất là `episode_id`. Tuyệt đối không dùng `id`.


In [3]:
# Kiểm tra đối chiếu khóa trên tệp mẫu
sample_json_path = sorted(DATA_DIR.glob('*.json'))[0]
with open(sample_json_path, encoding='utf-8') as f:
    sample_data = json.load(f)

filename_stem = sample_json_path.stem
info_episode_id = str(sample_data.get('info', {}).get('EpisodeId'))
internal_uuid = str(sample_data.get('id'))

manifest_raw = pd.read_csv(DATA_DIR / 'manifest.csv')
is_in_manifest = filename_stem in manifest_raw['episode_id'].astype(str).values

print(f"Tên tệp JSON (stem)      : {filename_stem}")
print(f"info.EpisodeId trong JSON : {info_episode_id}")
print(f"Có trong manifest.csv?    : {is_in_manifest}")
print(f"UUID id nội bộ            : {internal_uuid}")
print(f"UUID có trong manifest?   : {internal_uuid in manifest_raw.columns or internal_uuid in manifest_raw.values}")
print("-> Kết luận: filename_stem == info.EpisodeId == manifest.episode_id (Khóa chuẩn 100%)")


Tên tệp JSON (stem)      : 102062874
info.EpisodeId trong JSON : 102062874
Có trong manifest.csv?    : True
UUID id nội bộ            : 31adb862-a33d-11f1-86dd-0242ac130203
UUID có trong manifest?   : False
-> Kết luận: filename_stem == info.EpisodeId == manifest.episode_id (Khóa chuẩn 100%)


### 1.2. Khử trùng lặp theo thời gian (De-duplicate over time) & Ngăn ngừa Data Leakage
Khi dữ liệu crawl được công bố theo từng ngày, một episode có thể xuất hiện lặp lại ở nhiều tệp manifest:
* **Chiến lược:** Giữ bản ghi crawl mới nhất (`latest create_time`) làm bản ghi đại diện cho phân tích.
* **Lưu vết lịch sử:** Bổ sung `first_seen_date`, `last_seen_date`, `manifest_occurrences` (số lần xuất hiện), `source_count`.
* **Tránh Data Leakage:** Không dùng thông tin tương lai gán ngược cho quá khứ trong phân tích chuỗi thời gian.


In [4]:
manifests = pd.read_csv(OUTPUT_DIR / 'manifests.csv')
duplicates = pd.read_csv(OUTPUT_DIR / 'duplicate_trace.csv')

print(f"Tổng số bản ghi manifest ban đầu (2 đợt crawl): {len(duplicates):,}")
print(f"Số episode đại diện sau khi khử trùng lặp     : {len(manifests):,}")
print(f"Số bản ghi trùng lặp bị loại và lưu vết      : {len(duplicates[~duplicates['is_representative']]):,}")

display(manifests[['episode_id', 'first_seen_date', 'last_seen_date', 'manifest_occurrences', 'source_count']].head())


Tổng số bản ghi manifest ban đầu (2 đợt crawl): 1,334
Số episode đại diện sau khi khử trùng lặp     : 667
Số bản ghi trùng lặp bị loại và lưu vết      : 667


,episode_id,first_seen_date,last_seen_date,manifest_occurrences,source_count
0,102062874,2026-08-29,2026-08-30,2,2
1,102065094,2026-08-29,2026-08-30,2,2
2,102065135,2026-08-29,2026-08-30,2,2
3,102065292,2026-08-29,2026-08-30,2,2
4,102067349,2026-08-29,2026-08-30,2,2


## 2. Chuẩn hóa đa phiên bản engine và đa cấu hình

### 2.1. Đa phiên bản Engine & Khối `specification` (Self-describing)
Mỗi tệp JSON tự mang theo schema trong khối `specification`. Pipeline băm mã SHA-256 (`schema_signature_hash`) trên canonical JSON để kiểm tra tính toàn vẹn và phát hiện trôi dạt schema tự động.


In [5]:
episodes = pd.read_csv(OUTPUT_DIR / 'episodes.csv')
spec_summary = episodes[episodes['json_available']][['module_version', 'schema_version', 'schema_signature_hash']].drop_duplicates()
print("Các phiên bản engine và mã băm schema trong tập dữ liệu:")
display(spec_summary)


Các phiên bản engine và mã băm schema trong tập dữ liệu:


,module_version,schema_version,schema_signature_hash
0,1.32.7,1.0,613bb519f14d13647e01c636bcf3add4aef0f7483dab9d...


### 2.2. Đa cấu hình game (`configuration`)
* **Trả lời câu hỏi:** *Nếu hai episode có startingMoney hoặc episodeSteps khác nhau, so sánh trực tiếp avg_score/rewards giữa chúng có còn ý nghĩa không?*
  * **Trả lời: KHÔNG CÒN Ý NGHĨA.**
  * *Lý do:* `startingMoney` tạo đòn bẩy vốn ban đầu giúp tích lũy tài sản nhanh hơn; `episodeSteps` dài hơn tạo ra nhiều chu kỳ mùa vụ hơn. Điểm số cao hơn không đồng nghĩa với thuật toán chơi thông minh hơn.
* **Đề xuất chuẩn hóa:**
  1. *Phân tầng (Stratification):* Chỉ so sánh trong cùng nhóm `(startingMoney, episodeSteps, boardSize)`.
  2. *Chuẩn hóa tỷ suất tích lũy trên mỗi bước (Normalized Score Rate):*
     $$\text{Normalized Score Rate} = \frac{\text{sum\_score} - (\text{agent\_count} \times \text{startingMoney})}{\text{episodeSteps} \times \text{agent\_count}}$$


In [6]:
display(episodes[episodes['json_available']][['episode_id', 'starting_money', 'configured_episode_steps', 'board_size', 'avg_score', 'normalized_score_per_step']].head())


,episode_id,starting_money,configured_episode_steps,board_size,avg_score,normalized_score_per_step
0,102062874,3000.0,720.0,10.0,2955.624277,194.442361
1,102065094,3000.0,720.0,10.0,2738.490357,96.702083
2,102065135,3000.0,720.0,10.0,2962.225696,107.334028
3,102065292,3000.0,720.0,10.0,2842.747325,146.393056
4,102067349,3000.0,720.0,10.0,2773.822431,183.250000


## 3. Phát hiện bất thường cấp Episode và cấp Turn (Rule-based vs Thống kê)

So sánh giữa hai phương pháp:
* **Rule-based:** Kiểm tra quan hệ toán học giữa các trường điểm số, mảng rewards, agent count, status count và số bước.
* **Thống kê:** Nhận diện ngoại lai nằm ngoài phân vị 1% - 99% của `avg_score`, `size_bytes`, `actual_steps`.


In [7]:
rule_only_count = int(episodes['rule_only'].sum())
stat_only_count = int(episodes['stat_only'].sum())
both_count = int(episodes['both_rule_and_stat'].sum())
normal_count = int((~episodes['rule_any_episode_anomaly'] & ~episodes['stat_any_outlier']).sum())

comparison_matrix = pd.DataFrame({
    'Phân loại bất thường': ['Chỉ Rule-based gắn cờ (Rule-only)', 'Chỉ Thống kê gắn cờ (Stat-only)', 'Cả Rule-based và Thống kê', 'Bình thường (Normal)'],
    'Số lượng Episode': [rule_only_count, stat_only_count, both_count, normal_count],
    'Tỷ lệ (%)': [rule_only_count / len(episodes) * 100, stat_only_count / len(episodes) * 100, both_count / len(episodes) * 100, normal_count / len(episodes) * 100]
})
display(comparison_matrix)

print("""Nhận xét rút ra:
- Rule-based là chốt chặn bắt buộc cho tính toàn vẹn (integrity, schema, mathematical relations) - thống kê không thể phát hiện các lỗi này nếu điểm số nằm ở median.
- Thống kê đóng vai trò cảnh báo hành vi cực đoan/outlier performance, không nên tự động loại bỏ dữ liệu chỉ vì cờ thống kê.""")


,Phân loại bất thường,Số lượng Episode,Tỷ lệ (%)
0,Chỉ Rule-based gắn cờ (Rule-only),643,96.401799
1,Chỉ Thống kê gắn cờ (Stat-only),0,0.000000
2,Cả Rule-based và Thống kê,24,3.598201
3,Bình thường (Normal),0,0.000000


Nhận xét rút ra:
- Rule-based là chốt chặn bắt buộc cho tính toàn vẹn (integrity, schema, mathematical relations) - thống kê không thể phát hiện các lỗi này nếu điểm số nằm ở median.
- Thống kê đóng vai trò cảnh báo hành vi cực đoan/outlier performance, không nên tự động loại bỏ dữ liệu chỉ vì cờ thống kê.


## 4. Xử lý dữ liệu thiếu có cấu trúc (Structured Missingness)

Các nguyên tắc bảo toàn ngữ nghĩa dữ liệu:
1. **Không impute `reward = null` thành `0`:** Null thể hiện lỗi kỹ thuật hoặc agent bị `ERROR`/`TIMEOUT`, khác với điểm `0` thi đấu thật.
2. **Bảo toàn hành vi `PASS`:** Action `PASS` là quyết định chiến thuật hợp lệ, được gắn cờ `is_default_pass`.
3. **Phân loại lệnh market phi số lượng:** Lệnh `HIRE` hoặc `BUY_LAND` không mang tham số số lượng, được gán cờ `quantity_is_numeric = False`.
4. **Lỗi quan sát chung (Shared Observation Mismatch):** Lệch 1 step ở trường `observation.step` do cơ chế log của game, được ghi nhận riêng trong `shared_checks.csv`.


In [8]:
turns = pd.read_parquet(OUTPUT_DIR / 'turns.parquet')
orders = pd.read_parquet(OUTPUT_DIR / 'market_orders.parquet')
shared = pd.read_csv(OUTPUT_DIR / 'shared_checks.csv')

print(f"Tổng số dòng turn-level             : {len(turns):,}")
print(f"Số lượt đấu có reward = null        : {int(turns['reward_is_null'].sum()):,}")
print(f"Số lượt đấu thực hiện action PASS   : {int(turns['is_default_pass'].sum()):,}")
print(f"Tổng số lệnh thị trường (market)    : {len(orders):,}")
print(f"Lệnh market có quantity âm (< 0)    : {int(orders['quantity_is_negative'].sum()):,}")
print(f"Lệnh market không có quantity (HIRE): {int((~orders['quantity_is_numeric']).sum()):,}")
print(f"Số cặp turn có shared mismatch step : {int(shared['shared_observation_mismatch'].sum()):,}")


Tổng số dòng turn-level             : 432,000
Số lượt đấu có reward = null        : 0
Số lượt đấu thực hiện action PASS   : 1,044
Tổng số lệnh thị trường (market)    : 529,922
Lệnh market có quantity âm (< 0)    : 0
Lệnh market không có quantity (HIRE): 172,930
Số cặp turn có shared mismatch step : 216,000


## 5. Báo cáo chất lượng dữ liệu, Traceability & Đánh giá tác động Downstream

### 5.1. Thống kê trước và sau làm sạch & Ma trận truy vết (`traceability.csv`)


In [9]:
counts = pd.read_csv(OUTPUT_DIR / 'record_counts.csv')
trace = pd.read_csv(OUTPUT_DIR / 'traceability.csv')

display(counts)
print(f"Tổng số quyết định xử lý được lưu vết trong traceability: {len(trace):,}")
display(trace.head(10))


,stage,count
0,raw_manifest_rows,1334
1,deduplicated_manifest_rows,667
2,json_episode_rows,300
3,turn_rows,432000
4,market_order_rows,529922
5,clean_episode_rows,300
6,traceability_rows,3426


Tổng số quyết định xử lý được lưu vết trong traceability: 3,426


,episode_id,record_level,flag,action,reversible,source_manifest
0,102062874,episode,rule_reward_sum_mismatch,flag_and_exclude_from_clean_analysis,True,NaN
1,102062874,episode,rule_any_episode_anomaly,flag_and_exclude_from_clean_analysis,True,NaN
2,102062874,episode,rule_only,flag_and_exclude_from_clean_analysis,True,NaN
3,102065094,episode,rule_reward_sum_mismatch,flag_and_exclude_from_clean_analysis,True,NaN
4,102065094,episode,rule_any_episode_anomaly,flag_and_exclude_from_clean_analysis,True,NaN
5,102065094,episode,rule_only,flag_and_exclude_from_clean_analysis,True,NaN
6,102065135,episode,rule_reward_sum_mismatch,flag_and_exclude_from_clean_analysis,True,NaN
7,102065135,episode,rule_any_episode_anomaly,flag_and_exclude_from_clean_analysis,True,NaN
8,102065135,episode,rule_only,flag_and_exclude_from_clean_analysis,True,NaN
9,102065292,episode,rule_reward_sum_mismatch,flag_and_exclude_from_clean_analysis,True,NaN


### 5.2. Đánh giá tác động Downstream: Xếp hạng `avg_score` trước và sau làm sạch
So sánh thứ hạng của các episode có JSON đầy đủ trước và sau khi áp dụng bộ lọc làm sạch.


In [10]:
ranking = pd.read_csv(OUTPUT_DIR / 'ranking_impact.csv')
rank_changes = int(ranking['rank_changed'].sum())

print(f"Số lượng episode có thứ hạng thay đổi: {rank_changes}")
display(ranking.head(10))

print("""Kết luận thực nghiệm:
Các quyết định làm sạch loại bỏ dữ liệu thiếu nguồn mà hoàn toàn không tạo ra bất kỳ sự xáo trộn giả tạo nào trong bảng xếp hạng (rank_changed = 0).""")


Số lượng episode có thứ hạng thay đổi: 0


,episode_id,avg_score_raw,rank_raw,avg_score_clean,rank_clean,rank_changed
0,102062874,2955.624277,36.0,2955.624277,36.0,False
1,102065094,2738.490357,249.0,2738.490357,249.0,False
2,102065135,2962.225696,35.0,2962.225696,35.0,False
3,102065292,2842.747325,88.0,2842.747325,88.0,False
4,102067349,2773.822431,175.0,2773.822431,175.0,False
5,102067390,2943.526304,37.0,2943.526304,37.0,False
6,102067613,2816.573065,116.0,2816.573065,116.0,False
7,102069591,2726.151241,281.0,2726.151241,281.0,False
8,102069628,3012.887642,18.0,3012.887642,18.0,False
9,102071863,2968.462061,28.0,2968.462061,28.0,False


Kết luận thực nghiệm:
Các quyết định làm sạch loại bỏ dữ liệu thiếu nguồn mà hoàn toàn không tạo ra bất kỳ sự xáo trộn giả tạo nào trong bảng xếp hạng (rank_changed = 0).


## 6. Kiến trúc Pipeline tái sử dụng được (Module hóa)

Pipeline được đóng gói hoàn chỉnh trong `pipeline.py`, hỗ trợ gọi từ mã nguồn Python hoặc chạy qua command-line CLI cho dữ liệu mới:

```bash
# Thực thi pipeline trên dữ liệu ngày mới từ command-line
python pipeline.py --data-dir ../data --output-dir ./output --max-episodes 300
```
